# FAIR² Mental Health Survey Dataset Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json"

# Load dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}\n")
print(f"Version: {metadata.version}\n")
print(f"License: {metadata.license}\n")
print(f"Citation: {getattr(metadata, 'citeAs', None)}\n")

## 2. Data Overview
Review available record sets, fields, and their IDs (all `@id` values).

The Croissant schema provides one or more record sets, each with a unique `@id`. For this dataset, let's list all available record sets, the fields within each, and their `@id` attributes.

In [ ]:
# Enumerate record set IDs
record_sets = list(dataset.record_sets.keys())
print("Available record sets and their fields (by @id):\n")
for rs_id in record_sets:
    rs = dataset.record_sets[rs_id]
    print(f"Record set: {rs_id}")
    field_ids = [f.get('@id', f) if isinstance(f, dict) else f for f in getattr(rs, 'fields', [])]
    print(f"  Fields: {field_ids if field_ids else getattr(rs,'fields', None)}\n")

Let's print a few sample records from one of the main record sets (by `@id`).

In [ ]:
# Print first 2 records for each record set
for rs_id in record_sets:
    print(f"Sample records for record set: {rs_id}")
    for i, record in enumerate(dataset.records(record_set=rs_id)):
        if i >= 2:
            break
        print(record)
    print()

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. All references to record sets and fields use their `@id`.

In [ ]:
# Extract all records as DataFrames, organized by record set @id
dataframes = {}
for rs_id in record_sets:
    records = list(dataset.records(record_set=rs_id))
    if records:  # Non-empty
        dataframes[rs_id] = pd.DataFrame(records)
        print(f"Loaded DataFrame for {rs_id} with columns: {dataframes[rs_id].columns.tolist()}")

# Pick the first record set for further analysis as example
main_record_set_id = record_sets[0]
df_main = dataframes[main_record_set_id]
df_main.head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records, normalizing numeric fields, and grouping by categorical variables.

We'll use the field IDs as they appear in the DataFrame. For demonstration, let's select a likely numeric symptom score field if present (e.g. 'gad_7_score' or similar by `@id`).

In [ ]:
# List all columns to identify numeric fields by their @id
print(df_main.columns.tolist())

# Try to find a numeric field (by @id) for EDA, e.g. GAD-7, PHQ-9, or PSQ score
possible_numeric_fields = [c for c in df_main.columns if any(x in c.lower() for x in ['gad', 'phq', 'psq', 'score','age'])]
print(f"Possible numeric fields: {possible_numeric_fields}")

# Select by @id: If gad_7_score exists, use it. Else, pick from the list.
if 'gad_7_score' in df_main.columns:
    numeric_field_id = 'gad_7_score'
elif possible_numeric_fields:
    numeric_field_id = possible_numeric_fields[0]
else:
    numeric_field_id = df_main.columns[0]  # Fallback
print(f"Selected numeric field for analysis: {numeric_field_id}")

threshold = 10  # Example threshold
filtered_df = df_main[df_main[numeric_field_id] > threshold]
print(f"Filtered records with {numeric_field_id} > {threshold}:")
print(filtered_df.head())

filtered_df[f"{numeric_field_id}_normalized"] = (
    filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
) / filtered_df[numeric_field_id].std()

print(f"Normalized {numeric_field_id} for filtered records:")
print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Try grouping by a categorical variable, e.g., 'gender' or 'marital_status' if present
group_fields = [c for c in df_main.columns if any(x in c.lower() for x in ['gender', 'marital', 'village', 'education'])]
if group_fields:
    group_field = group_fields[0]
    grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean()
    print(f"Grouped data by {group_field} (mean of {numeric_field_id}):")
    print(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of the selected numeric field
plt.figure(figsize=(7,4))
sns.histplot(df_main[numeric_field_id].dropna(), bins=20, kde=True)
plt.title(f"Distribution of {numeric_field_id}")
plt.xlabel(numeric_field_id)
plt.ylabel("Count")
plt.show()

# Boxplot by group field (if available)
if group_fields:
    plt.figure(figsize=(8,5))
    sns.boxplot(x=df_main[group_field], y=df_main[numeric_field_id])
    plt.title(f"{numeric_field_id} by {group_field}")
    plt.ylabel(numeric_field_id)
    plt.xlabel(group_field)
    plt.show()

## 6. Conclusion
In this notebook, we've loaded and explored the FAIR² Mental Health Survey Dataset using the `mlcroissant` library. We've provided a record set overview, loaded records by their `@id`, performed filtering and normalization on a key measurement field, and visualized the data distributions and group comparisons.

Further analysis can be performed depending on research questions, such as correlational analysis between psychometric scores, demographic attributes, or time-based trends if present.